In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import networkx as nx
import torch
from collections import Counter

from config import load_project_config_json, UDGConfig
from graphs.unit_disk import generate_square_lattice_udg
from module1.policy import SchedulePolicy
from module1.base import FixedScheduleBaseline
from module2.braket_backend import BraketBackend
from module2.interfaces import BackendResult

In [ ]:
cfg = load_project_config_json()
print(f"Backend:       {cfg.backend}")
print(f"Architecture:  {cfg.controls.architecture}")
print(f"learn_omega:   {cfg.controls.learn_omega}")
print(f"N_t:           {cfg.controls.N_t}")
print(f"T:             {cfg.controls.T:.2e} s  ({cfg.controls.T*1e6:.1f} \u00b5s)")
print(f"UDG grid:      {cfg.udg.nx}x{cfg.udg.ny}, spacing={cfg.udg.spacing}, r={cfg.udg.radius}")
print(f"Dropout:       {cfg.udg.dropout_rate}, seed={cfg.udg.seed}")
print(f"Hardware:      C6={cfg.hardware.C6:.2e}, \u03a9_max={cfg.hardware.omega_max}, t_ramp={cfg.hardware.t_ramp}, t_onset={cfg.hardware.t_onset}")

## 1. Generate two UDGs and produce schedules

In [ ]:
seeds = [cfg.udg.seed, cfg.udg.seed + 1]
graphs, positions = [], []

for s in seeds:
    udg_cfg = UDGConfig(
        nx=cfg.udg.nx, ny=cfg.udg.ny, spacing=cfg.udg.spacing,
        radius=cfg.udg.radius, dropout_rate=cfg.udg.dropout_rate, seed=s,
    )
    G, pos = generate_square_lattice_udg(udg_cfg)
    graphs.append(G)
    positions.append(pos)
    mis_approx = nx.algorithms.approximation.maximum_independent_set(G)
    print(f"seed={s}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, approx MIS size={len(mis_approx)}")

In [ ]:
# Build both a learned policy (untrained) and the fixed baseline
torch.manual_seed(42)
policy = SchedulePolicy(cfg)
baseline = FixedScheduleBaseline(cfg)

# Produce schedules from both
learned_schedules = [policy.make_schedule(G) for G in graphs]
baseline_schedule = baseline.make_schedule(graphs[0])  # same for all graphs

print(f"Learned policy:  {sum(p.numel() for p in policy.parameters()):,} params")
print(f"Baseline:        fixed trapezoidal \u03a9 + linear \u0394 sweep")

## 2. Visualize the input schedules

Side-by-side comparison of the baseline (Ebadi-style) and the untrained learned policy schedules.

In [ ]:
t_us = np.linspace(0, cfg.controls.T * 1e6, cfg.controls.N_t)
colors = ["#ef4444", "#3b82f6"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

# Baseline
axes[0, 0].plot(t_us, baseline_schedule.omega, color="#6b7280", linewidth=2)
axes[0, 0].set_ylabel("\u03a9(t) [rad/\u00b5s]", fontsize=12)
axes[0, 0].set_title("Baseline (fixed adiabatic)", fontsize=13, fontweight="bold")
axes[0, 0].grid(alpha=0.3)

axes[1, 0].plot(t_us, baseline_schedule.delta, color="#6b7280", linewidth=2)
axes[1, 0].set_ylabel("\u0394(t) [rad/\u00b5s]", fontsize=12)
axes[1, 0].set_xlabel("Time [\u00b5s]", fontsize=12)
axes[1, 0].grid(alpha=0.3)

# Learned (untrained)
for sched, s, c in zip(learned_schedules, seeds, colors):
    axes[0, 1].plot(t_us, sched.omega, color=c, linewidth=1.5, label=f"seed {s}")
    axes[1, 1].plot(t_us, sched.delta, color=c, linewidth=1.5, label=f"seed {s}")

axes[0, 1].set_title("Learned policy (untrained)", fontsize=13, fontweight="bold")
axes[0, 1].set_ylabel("\u03a9(t) [rad/\u00b5s]", fontsize=12)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(alpha=0.3)

axes[1, 1].set_ylabel("\u0394(t) [rad/\u00b5s]", fontsize=12)
axes[1, 1].set_xlabel("Time [\u00b5s]", fontsize=12)
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(alpha=0.3)

fig.suptitle("Module 1 \u2192 Module 2: Input Schedules", fontsize=15, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

## 3. Run the Braket local simulator

We feed the schedules from both the baseline and learned policy through `BraketBackend` and collect measurement results.

In [ ]:
N_SHOTS = 100

backend = BraketBackend(cfg, n_shots=N_SHOTS, backend_type="simulator", validate=True)

results = {}  # (source, seed) -> BackendResult

for i, (G, pos, s) in enumerate(zip(graphs, positions, seeds)):
    # Baseline schedule
    r_base = backend.estimate_p_mis(baseline_schedule, G, pos)
    results[("baseline", s)] = r_base
    print(f"Baseline  seed={s}: p_MIS = {r_base.p_mis:.2%} \u00b1 {r_base.std_err:.2%}  ({r_base.shots} shots)")

    # Learned schedule
    r_learn = backend.estimate_p_mis(learned_schedules[i], G, pos)
    results[("learned", s)] = r_learn
    print(f"Learned   seed={s}: p_MIS = {r_learn.p_mis:.2%} \u00b1 {r_learn.std_err:.2%}  ({r_learn.shots} shots)")
    print()

## 4. p_MIS comparison: baseline vs. learned

Bar chart comparing p_MIS from the fixed adiabatic baseline and the (untrained) learned policy.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(len(seeds))
width = 0.35

base_vals = [results[("baseline", s)].p_mis for s in seeds]
base_errs = [results[("baseline", s)].std_err for s in seeds]
learn_vals = [results[("learned", s)].p_mis for s in seeds]
learn_errs = [results[("learned", s)].std_err for s in seeds]

ax.bar(x - width/2, base_vals, width, yerr=base_errs,
       label="Baseline (fixed)", color="#6b7280", capsize=4, alpha=0.85)
ax.bar(x + width/2, learn_vals, width, yerr=learn_errs,
       label="Learned (untrained)", color="#3b82f6", capsize=4, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f"seed {s}" for s in seeds], fontsize=11)
ax.set_ylabel("p_MIS", fontsize=13)
ax.set_title("Module 2 Output: p_MIS by Schedule Source", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(bottom=0)

fig.tight_layout()
plt.show()

## 5. Measurement-outcome analysis

For one graph, break down the raw bitstring counts: how many Rydberg excitations per shot, and which bitstrings form independent sets.

In [ ]:
from module2.graph_MIS_utils import check_independent_set

# Pick the baseline result for the first graph
focus_seed = seeds[0]
focus_result = results[("baseline", focus_seed)]
focus_G = graphs[0]
counts = focus_result.counts

print(f"Graph seed={focus_seed}: {focus_G.number_of_nodes()} nodes, {focus_G.number_of_edges()} edges")
print(f"Total unique bitstrings measured: {len(counts)}")
print(f"Total shots: {sum(counts.values())}")
print()

In [ ]:
# Rydberg count distribution: how many atoms are excited per shot?
rydberg_counts = {}
is_counts = {"IS": 0, "Not IS": 0}

for bs, n in counts.items():
    n_ryd = bs.count('r')
    rydberg_counts[n_ryd] = rydberg_counts.get(n_ryd, 0) + n
    if check_independent_set(bs, focus_G):
        is_counts["IS"] += n
    else:
        is_counts["Not IS"] += n

# Approximate MIS size
mis_approx = nx.algorithms.approximation.maximum_independent_set(focus_G)
mis_size = len(mis_approx)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# --- Panel 1: Rydberg count histogram ---
max_ryd = max(rydberg_counts.keys())
xs = list(range(max_ryd + 1))
ys = [rydberg_counts.get(k, 0) for k in xs]
bar_colors = ["#10b981" if k == mis_size else "#94a3b8" for k in xs]

axes[0].bar(xs, ys, color=bar_colors, edgecolor="white", linewidth=0.5)
axes[0].axvline(mis_size, color="#ef4444", linestyle="--", linewidth=1.5, label=f"MIS size \u2248 {mis_size}")
axes[0].set_xlabel("# Rydberg excitations", fontsize=11)
axes[0].set_ylabel("Shots", fontsize=11)
axes[0].set_title("Rydberg count distribution", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(axis="y", alpha=0.3)

# --- Panel 2: IS vs non-IS pie chart ---
is_labels = list(is_counts.keys())
is_values = list(is_counts.values())
is_colors = ["#10b981", "#f87171"]
axes[1].pie(
    is_values, labels=is_labels, colors=is_colors,
    autopct="%1.0f%%", startangle=90, textprops={"fontsize": 11},
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
axes[1].set_title("Independent set fraction", fontsize=12, fontweight="bold")

# --- Panel 3: Top bitstrings by frequency ---
sorted_bs = sorted(counts.items(), key=lambda x: -x[1])[:12]
bs_labels = [bs[:10] + ("\u2026" if len(bs) > 10 else "") for bs, _ in sorted_bs]
bs_vals = [n for _, n in sorted_bs]
bs_colors_list = ["#10b981" if check_independent_set(bs, focus_G) else "#f87171" for bs, _ in sorted_bs]

axes[2].barh(range(len(bs_labels)), bs_vals, color=bs_colors_list, edgecolor="white")
axes[2].set_yticks(range(len(bs_labels)))
axes[2].set_yticklabels(bs_labels, fontsize=8, fontfamily="monospace")
axes[2].invert_yaxis()
axes[2].set_xlabel("Shots", fontsize=11)
axes[2].set_title("Top bitstrings (\u2705 IS / \u274c not IS)", fontsize=12, fontweight="bold")
axes[2].grid(axis="x", alpha=0.3)

fig.suptitle(f"Measurement Analysis \u2014 Baseline, seed={focus_seed}, {N_SHOTS} shots",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 6. Visualize the graph with MIS solutions

Overlay the most-frequently-measured MIS solution (if any) onto the atom arrangement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, s, G, pos in zip(axes, seeds, graphs, positions):
    result = results[("baseline", s)]
    mis_bitstrings = result.metadata.get("mis_bitstrings", [])

    if mis_bitstrings:
        # Pick the most frequently measured MIS bitstring
        best_bs = max(mis_bitstrings, key=lambda bs: result.counts.get(bs, 0))
        mis_nodes = {i for i, c in enumerate(best_bs) if c == 'r'}
        subtitle = f"p_MIS={result.p_mis:.1%}, |MIS|={len(mis_nodes)}"
    else:
        mis_nodes = set()
        subtitle = f"p_MIS=0% (no MIS found in {N_SHOTS} shots)"

    node_list = sorted(G.nodes())
    node_colors = ["#ef4444" if n in mis_nodes else "#3b82f6" for n in node_list]
    node_sizes = [120 if n in mis_nodes else 50 for n in node_list]

    nx.draw(
        G, pos, ax=ax, with_labels=False,
        nodelist=node_list, node_color=node_colors, node_size=node_sizes,
        edge_color="#cbd5e1", width=0.8, alpha=0.9,
    )
    ax.set_title(f"seed {s} \u2014 {subtitle}", fontsize=12, fontweight="bold")

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#ef4444',
               markersize=10, label='MIS atom (Rydberg)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#3b82f6',
               markersize=8, label='Ground state'),
    ]
    ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

fig.suptitle("Atom Arrangements with Best Measured MIS Solution",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

## 7. End-to-end summary

Compact dashboard: graph topology, input schedules, and output p_MIS in one view per graph.

In [ ]:
fig = plt.figure(figsize=(16, 10))
outer = gridspec.GridSpec(2, 1, hspace=0.35, figure=fig)

for row, (G, pos, sched, s) in enumerate(zip(graphs, positions, learned_schedules, seeds)):
    inner = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=outer[row],
                                            wspace=0.35, hspace=0.3,
                                            width_ratios=[1, 1.2, 1.2])

    r_base = results[("baseline", s)]
    r_learn = results[("learned", s)]

    # --- Col 0: Graph ---
    ax_graph = fig.add_subplot(inner[:, 0])
    mis_approx = nx.algorithms.approximation.maximum_independent_set(G)
    nc = ["#ef4444" if n in mis_approx else "#3b82f6" for n in sorted(G.nodes())]
    nx.draw(G, pos, ax=ax_graph, with_labels=False,
            node_color=nc, node_size=60, edge_color="#cbd5e1", width=0.6)
    ax_graph.set_title(f"seed {s}\n{G.number_of_nodes()}n, {G.number_of_edges()}e, |MIS|\u2248{len(mis_approx)}",
                       fontsize=11, fontweight="bold")

    # --- Col 1: Omega ---
    ax_omega = fig.add_subplot(inner[0, 1])
    ax_omega.plot(t_us, baseline_schedule.omega, color="#6b7280", linewidth=1.5, label="baseline", linestyle="--")
    ax_omega.plot(t_us, sched.omega, color=colors[row], linewidth=1.5, label="learned")
    ax_omega.set_ylabel("\u03a9 [rad/\u00b5s]", fontsize=10)
    ax_omega.set_title("\u03a9(t)", fontsize=11, fontweight="bold")
    ax_omega.legend(fontsize=8)
    ax_omega.grid(alpha=0.3)

    # --- Col 1 bottom: Delta ---
    ax_delta = fig.add_subplot(inner[1, 1], sharex=ax_omega)
    ax_delta.plot(t_us, baseline_schedule.delta, color="#6b7280", linewidth=1.5, label="baseline", linestyle="--")
    ax_delta.plot(t_us, sched.delta, color=colors[row], linewidth=1.5, label="learned")
    ax_delta.set_ylabel("\u0394 [rad/\u00b5s]", fontsize=10)
    ax_delta.set_xlabel("Time [\u00b5s]", fontsize=10)
    ax_delta.set_title("\u0394(t)", fontsize=11, fontweight="bold")
    ax_delta.legend(fontsize=8)
    ax_delta.grid(alpha=0.3)

    # --- Col 2: p_MIS comparison ---
    ax_pmis = fig.add_subplot(inner[:, 2])
    bar_x = [0, 1]
    bar_vals = [r_base.p_mis, r_learn.p_mis]
    bar_errs = [r_base.std_err, r_learn.std_err]
    bar_colors = ["#6b7280", colors[row]]
    ax_pmis.bar(bar_x, bar_vals, yerr=bar_errs, color=bar_colors,
               capsize=6, width=0.6, edgecolor="white", linewidth=1)
    ax_pmis.set_xticks(bar_x)
    ax_pmis.set_xticklabels(["Baseline", "Learned"], fontsize=10)
    ax_pmis.set_ylabel("p_MIS", fontsize=11)
    ax_pmis.set_title("Output: p_MIS", fontsize=11, fontweight="bold")
    ax_pmis.set_ylim(bottom=0)
    ax_pmis.grid(axis="y", alpha=0.3)
    for j, (v, e) in enumerate(zip(bar_vals, bar_errs)):
        ax_pmis.text(j, v + e + 0.01, f"{v:.1%}", ha="center", fontsize=10, fontweight="bold")

fig.suptitle("End-to-End Pipeline: Graph \u2192 Module 1 (Schedule) \u2192 Module 2 (p_MIS)",
             fontsize=15, fontweight="bold", y=1.01)
plt.show()